In [ ]:
Oracle AI Data Platform v1.0

Copyright © 2025, Oracle and/or its affiliates.

Licensed under the Universal Permissive License v 1.0 as shown at https://oss.oracle.com/licenses/upl/

# Autonomous AI Lakehouse Connector Samples

Read/write ingestion, external-catalog, and pushdown samples for the `ORACLE_ALH` connector.


## Ingestion Samples

Use inline connector options when you want the notebook to carry all connection properties explicitly.


In [ ]:
wallet_content = "<BASE64_ENCODED_WALLET>"

# Ingestion-style read using inline connector options
alh_df = spark.read.format("aidataplatform") \
    .option("type", "ORACLE_ALH") \
    .option("wallet.content", wallet_content) \
    .option("tns", "<TNS_ALIAS>") \
    .option("user.name", "<USERNAME>") \
    .option("password", "<PASSWORD>") \
    .option("schema", "<SCHEMA>") \
    .option("table", "<TABLE_NAME>") \
    .load()

alh_df.show()


## Wallet Path Samples

Use `wallet.path` when the wallet zip is already available in a workspace path or mounted volume instead of passing Base64 wallet content inline.


### Using Workspace Wallet Path

This option reads the ALH wallet from a workspace location.


In [ ]:
df = (
    spark.read.format("aidataplatform")
    .option("type", "ORACLE_ALH")
    .option("wallet.path", "/Workspace/Wallet_customer_alh.zip")
    .option("tns", "testdev_high")
    .option("user.name", "ADMIN")
    .option("password", "PASSWORD")
    .option("schema", "SCHEMA")
    .option("table", "TABLE")
    .load()
)

df.show()


### Using Volumes Wallet Path

This option reads the ALH wallet from a mounted volume.


In [ ]:
df = (
    spark.read.format("aidataplatform")
    .option("type", "ORACLE_ALH")
    .option("wallet.path", "/Volumes/default/test/Wallet_customer_alh.zip")
    .option("tns", "testdev_high")
    .option("user.name", "ADMIN")
    .option("password", "PASSWORD")
    .option("schema", "SCHEMA")
    .option("table", "TABLE")
    .load()
)

df.show()


In [ ]:
# Ingestion-style write using inline connector options
alh_df.write.format("aidataplatform") \
    .option("type", "ORACLE_ALH") \
    .option("wallet.content", wallet_content) \
    .option("tns", "<TNS_ALIAS>") \
    .option("user.name", "<USERNAME>") \
    .option("password", "<PASSWORD>") \
    .option("schema", "<SCHEMA>") \
    .option("table", "<TARGET_TABLE_NAME>") \
    .option("write.mode", "APPEND") \
    .save()


## Ingestion Samples Using `catalog.id`

Use `catalog.id` to reuse an existing external catalog connection and avoid placing credentials directly in the notebook.


In [ ]:
# Ingestion-style read using an existing external catalog connection
alh_df_catalog = spark.read.format("aidataplatform") \
    .option("catalog.id", "<CATALOG_ID>") \
    .option("schema", "<SCHEMA>") \
    .option("table", "<TABLE_NAME>") \
    .load()

alh_df_catalog.show()

# Ingestion-style write using an existing external catalog connection
alh_df.write.format("aidataplatform") \
    .option("catalog.id", "<CATALOG_ID>") \
    .option("schema", "<SCHEMA>") \
    .option("table", "<TARGET_TABLE_NAME>") \
    .option("write.mode", "APPEND") \
    .save()


## External Catalog Samples

After creating an external catalog in Master Catalogs, use three-part names in the form `<CATALOG_ID>.<SCHEMA>.<TABLE_NAME>` for Spark table access.


In [ ]:
# External catalog read with three-part name
catalog_df = spark.table("<CATALOG_ID>.<SCHEMA>.<TABLE_NAME>")
catalog_df.show()

# External catalog write - create
catalog_df.write.saveAsTable("<CATALOG_ID>.<SCHEMA>.<TARGET_TABLE_NAME>")

# External catalog write - overwrite
catalog_df.write.mode("overwrite").saveAsTable("<CATALOG_ID>.<SCHEMA>.<TARGET_TABLE_NAME>")

# External catalog write - append
catalog_df.write.mode("append").insertInto("<CATALOG_ID>.<SCHEMA>.<TARGET_TABLE_NAME>")

# External catalog write - merge
catalog_df.write     .option("write.mode", "MERGE")     .option("write.merge.keys", "<KEY_COLUMN>")     .insertInto("<CATALOG_ID>.<SCHEMA>.<TARGET_TABLE_NAME>")


## Pushdown Samples

Use these samples when the source should execute filters or full SQL instead of Spark reading the full table first.


In [ ]:
# Auto pushdown from DataFrame operations
alh_df_filtered = spark.read.format("aidataplatform") \
    .option("type", "ORACLE_ALH") \
    .option("wallet.content", wallet_content) \
    .option("tns", "<TNS_ALIAS>") \
    .option("user.name", "<USERNAME>") \
    .option("password", "<PASSWORD>") \
    .option("schema", "<SCHEMA>") \
    .option("table", "<TABLE_NAME>") \
    .option("row.limit", "1000") \
    .load() \
    .select("<COLUMN_1>", "<COLUMN_2>") \
    .filter("<COLUMN_1> = '<VALUE>'")

alh_df_filtered.show()


In [ ]:
wallet_content = "<BASE64_ENCODED_WALLET>"

# Oracle SQL pushdown using inline connector options
alh_df_pushdown = spark.read.format("aidataplatform") \
    .option("type", "ORACLE_ALH") \
    .option("wallet.content", wallet_content) \
    .option("tns", "<TNS_ALIAS>") \
    .option("user.name", "<USERNAME>") \
    .option("password", "<PASSWORD>") \
    .option("pushdown.sql", "SELECT * FROM <SCHEMA>.<TABLE_NAME> WHERE <COLUMN_NAME> = '<VALUE>'") \
    .load()

alh_df_pushdown.show()


In [ ]:
# Oracle SQL pushdown using an external catalog connection
alh_df_catalog_pushdown = spark.read.format("aidataplatform") \
    .option("catalog.id", "<CATALOG_ID>") \
    .option("pushdown.sql", "SELECT * FROM <SCHEMA>.<TABLE_NAME> WHERE <COLUMN_NAME> = '<VALUE>'") \
    .load()

alh_df_catalog_pushdown.show()


## ALH Pushdown Options

| Option | Support | Example | Notes |
| --- | --- | --- | --- |
| Ingestion auto pushdown | Projections, filters, and limits through DataFrame operations | `.select(...).filter(...).option("row.limit", "1000")` | Joins and complex functions depend on Spark/source support. |
| Ingestion full SQL pushdown | Source SQL through `pushdown.sql` | `.option("pushdown.sql", "SELECT ...")` | Prefer Oracle SQL when you need precise source semantics. |
| Ingestion with `catalog.id` | Reuses an external catalog connection | `.option("catalog.id", "<CATALOG_ID>").option("pushdown.sql", "SELECT ...")` | Avoids embedding credentials in notebooks. |
| External catalog DataFrame pushdown | Aggregation, projection, filter, limit, offset, and TopN where Spark delegates them | `spark.read.table("catalog.schema.table").filter(...)` | `spark.sql` is not the recommended path for source pushdown. |


In [ ]:
# External catalog pushdown through Spark DataFrame operations
catalog_df = spark.read.table("<CATALOG_ID>.<SCHEMA>.<TABLE_NAME>")

catalog_df     .select("<COLUMN_1>", "<COLUMN_2>")     .filter("<COLUMN_1> = '<VALUE>'")     .orderBy("<COLUMN_2>")     .limit(100)     .show()

# Aggregate pushdown is supported for external catalog reads where Spark delegates it.
catalog_df.groupBy("<COLUMN_1>").count().show()


## Connector Options

| Parameter name | Valid values | Mandatory | Description |
| --- | --- | --- | --- |
| `type` | `ORACLE_ALH` | Y for inline connector config | Data source type. |
| `catalog.id` | Existing external catalog identifier | N | Reuse the connection details from an already created external catalog instead of passing the connector connection properties inline. |
| `wallet.content` | Base64-encoded wallet content | Y, unless `wallet.path` is used | Wallet zip content for ingestion-style connectivity. |
| `wallet.path` | `/Workspace/<path>` or `/Volumes/<path>` | N | Alternate wallet location when wallet content is not passed inline. |
| `wallet.password` | Wallet password | N | Optional wallet password. |
| `tns` | `<service>_high`, `<service>_medium`, `<service>_low` | Y | TNS alias or service name to connect to the database. |
| `user.name` | Valid DB user | Y | Login user used for connectivity. |
| `password` | Valid DB password | Y | Password for `user.name`. |
| `oracle.user.name.quoted` | `true`, `false` | N | Use this when the Oracle login user itself was created as a quoted mixed-case or lower-case username. |
| `schema` | Valid schema name | Y | Schema to read from or write to. |
| `table` | Valid table name | Y for table-based access | Table name inside the schema. |
| `view` | Valid view name | N | View name inside the schema when reading through a view. |
| `sql` | Valid SQL query | N | Custom SQL query for read scenarios. |
| `fetch.size` | Positive integer | N | Number of records fetched per round trip for reads. |
| `row.limit` | Positive integer | N | Limits the number of rows read. |
| `partition.column` | Column name | N | Column used for range-based partitioned reads or writes. |
| `partition.num` | Integer >= 1 | N | Number of partitions to use for range partitioning. |
| `partition.lower` | Numeric value or ISO-8601 date/time | N | Lower bound for range partitioning. |
| `partition.upper` | Numeric value or ISO-8601 date/time | N | Upper bound for range partitioning. |
| `pushdown.columns` | Comma-separated column names | N | Restricts the read to only the requested columns. |
| `pushdown.filter` | SQL predicate | N | Predicate pushed down to the source. |
| `pushdown.sql` | SQL query | N | Full SQL pushdown query. |
| `source.projection.pushdown.disabled` | `true`, `false` | N | Disable projection pushdown for external-catalog reads. |
| `source.filter.pushdown.disabled` | `true`, `false` | N | Disable filter pushdown for external-catalog reads. |
| `source.topn.pushdown.disabled` | `true`, `false` | N | Disable TopN pushdown for external-catalog reads. |
| `source.aggregation.pushdown.disabled` | `true`, `false` | N | Disable aggregation pushdown for external-catalog reads. |
| `source.limit.pushdown.disabled` | `true`, `false` | N | Disable limit pushdown for external-catalog reads. |
| `source.offset.pushdown.disabled` | `true`, `false` | N | Disable offset pushdown for external-catalog reads. |
| `write.mode` | `CREATE`, `APPEND`, `OVERWRITE`, `MERGE` | Y during write | Write mode to use. This is primarily used for ingestion samples; refer to the sample code above for external catalog write examples. |
| `write.empty.value.as.null` | `true`, `false` | N | Convert empty incoming string values to null before writing. |
| `write.batch.size` | Positive integer | N | Number of rows written in a batch. |
| `write.reject.limit` | Non-negative integer | N | Maximum rejected-row threshold before the write fails. |
| `write.merge.keys` | Comma-separated column names | Y for `MERGE` | Key columns used in the `MERGE` condition. |
| `merge.matched.where` | SQL condition using `tgt` and `src` aliases | N | Additional filter applied to rows where the `MERGE` match condition succeeds. |
| `merge.matched.delete` | SQL condition using `tgt` and `src` aliases | N | Delete matching target rows when the given condition evaluates to true. |
| `merge.not.matched.where` | SQL condition using `tgt` and `src` aliases | N | Additional filter applied before inserting rows that do not match. |
| `merge.staging.using.username` | `true`, `false` | N | For `MERGE`, stage the temporary merge table in the login user's schema from `user.name` instead of the default target schema. |
| `overwrite.with.recreate` | `true`, `false` | N | Controls overwrite behavior when the target shape is incompatible and overwrite needs a drop-and-recreate. |
| `skip.oos.staging` | `true`, `false` | N | Use this request-scoped override when you want the write to bypass object-storage staging for just this operation instead of relying on the older environment-wide flag. |
| `oracle.write.native.boolean` | `true`, `false` | N | Write Spark boolean columns as native Oracle `BOOLEAN` when the target Oracle version supports it. Leave it unset to keep the backward-compatible non-native path. |
| `oracle.append.hint.enabled` | `true`, `false` | N | For append writes, request Oracle append-insert behavior when you want the target write path to honor the append hint. |
| `preserve.oracle.column.types` | Comma-separated column/type mappings such as `EMBEDDING VECTOR(512, FLOAT32)`, `DOC JSON`, `RAW_PAYLOAD RAW(16)` | N | Create-only Oracle option to preserve Oracle-native target types during table creation instead of falling back to generic mapped types. Use this when the new table must keep special Oracle types such as `VECTOR`, `JSON`, or `RAW`. |
| `login.timeout.seconds` | Integer from `1` to `300` | N | JDBC login timeout in seconds. |
